## CA-CBAM + Focal Loss — YOLOv8s Backbone (v2 data)

Same `CoordinateCBAM`-augmented YOLOv8s backbone as `ca-cbam-on-v2-data.ipynb`, with the classification loss additionally replaced by focal loss (gamma=2.0, alpha=0.25) to test whether stronger down-weighting of easy negatives helps on the harder/minority classes. Box and DFL losses are left unchanged. Same training config: 100 epochs, imgsz 640, batch 16, patience 20, seed 42.

In [ ]:
# install ultralytics
!pip install ultralytics

In [ ]:
# import libraries and set seed
import os
import random
import yaml
import torch
import torch.nn as nn

from ultralytics import YOLO

SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Ultralytics:", __import__("ultralytics").__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

### Custom attention modules: Coordinate Attention + CBAM

In [ ]:
# create channelAttention class
class ChannelAttention(nn.Module):

    def __init__(self, channels, reduction=16):
        super().__init__()

        hidden = max(channels // reduction, 1)

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1, bias=False)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg = self.mlp(self.avg_pool(x))
        max_val = self.mlp(self.max_pool(x))

        return self.sigmoid(avg + max_val)

In [ ]:
# create spatialAttention class
class SpatialAttention(nn.Module):

    def __init__(self, kernel_size=7):
        super().__init__()

        padding = kernel_size // 2

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size,
            padding=padding,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg = torch.mean(x, dim=1, keepdim=True)
        max_val, _ = torch.max(x, dim=1, keepdim=True)

        attention = torch.cat(
            [avg, max_val],
            dim=1
        )

        return self.sigmoid(self.conv(attention))

In [ ]:
# create coordinate attention class
class CoordinateAttention(nn.Module):

    def __init__(self, channels, reduction=32):
        super().__init__()

        hidden = max(channels // reduction, 8)

        self.conv1 = nn.Conv2d(
            channels,
            hidden,
            kernel_size=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(hidden)
        self.act = nn.Hardswish()

        self.conv_h = nn.Conv2d(
            hidden,
            channels,
            kernel_size=1,
            bias=False
        )

        self.conv_w = nn.Conv2d(
            hidden,
            channels,
            kernel_size=1,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        identity = x

        b, c, h, w = x.size()

        # Pool along width
        x_h = torch.mean(
            x,
            dim=3,
            keepdim=True
        )

        # Pool along height
        x_w = torch.mean(
            x,
            dim=2,
            keepdim=True
        )

        x_w = x_w.permute(
            0, 1, 3, 2
        )

        # Concatenate coordinate information
        y = torch.cat(
            [x_h, x_w],
            dim=2
        )

        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)

        # Split height and width branches
        x_h, x_w = torch.split(
            y,
            [h, w],
            dim=2
        )

        x_w = x_w.permute(
            0, 1, 3, 2
        )

        a_h = self.sigmoid(
            self.conv_h(x_h)
        )

        a_w = self.sigmoid(
            self.conv_w(x_w)
        )

        return identity * a_h * a_w

In [ ]:
# combine coordinate, channel and spatial attention into CoordinateCBAM
class CoordinateCBAM(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.coordinate = CoordinateAttention(
            channels
        )

        self.channel = ChannelAttention(
            channels
        )

        self.spatial = SpatialAttention()

    def forward(self, x):

        # Coordinate attention
        x = self.coordinate(x)

        # Channel attention
        x = x * self.channel(x)

        # Spatial attention
        x = x * self.spatial(x)

        return x

### Register the module & build the modified YOLOv8s architecture

In [ ]:
# register CoordinateCBAM with ultralytics
import ultralytics.nn.tasks as tasks

# Register the custom module so the YAML architecture parser can resolve
# "CoordinateCBAM" by name when it builds the model from the YAML below.
tasks.CoordinateCBAM = CoordinateCBAM

print("CoordinateCBAM registered with Ultralytics.")

In [ ]:
# define CA-CBAM YOLOv8s architecture (YAML)
v4_yaml = """
nc: 17

depth_multiple: 0.33
width_multiple: 0.50

backbone:

  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]

  # CA-CBAM
  - [-1, 1, CoordinateCBAM, [128]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]

  # CA-CBAM
  - [-1, 1, CoordinateCBAM, [256]]

  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]

  # CA-CBAM
  - [-1, 1, CoordinateCBAM, [512]]

  - [-1, 1, SPPF, [1024, 5]]

head:

  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 8], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]

  - [-1, 1, nn.Upsample, [None, 2, nearest]]
  - [[-1, 5], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]

  - [[18, 21, 24], 1, Detect, [nc]]
"""

v4_yaml_path = "/kaggle/working/yolov8s_ca_cbam.yaml"

with open(v4_yaml_path, "w") as f:
    f.write(v4_yaml)

print("Saved:", v4_yaml_path)

In [ ]:
# verify yaml file was saved
from pathlib import Path

yaml_file = Path("/kaggle/working/yolov8s_ca_cbam.yaml")

print("Exists:", yaml_file.exists())
print("Path:", yaml_file)

### Build & sanity-check the model

In [ ]:
# build the model from the custom architecture
model_v4 = YOLO(
    "/kaggle/working/yolov8s_ca_cbam.yaml"
)

print("V4 model created successfully.")

In [ ]:
# print model architecture
print(model_v4.model)

In [ ]:
# prepare v2 dataset yaml for kaggle
import shutil
import yaml
import os

# V2 dataset YAML (read-only Kaggle input)
source_yaml = "/kaggle/input/datasets/jharshin/v2-enhanced-dataset/enhanced_yolo_dataset_v2/data.yaml"

# New writable YAML for this session
v2_yaml = "/kaggle/working/datav2.yaml"

# Copy the original YAML
shutil.copy(source_yaml, v2_yaml)

# Load it
with open(v2_yaml, "r") as f:
    data = yaml.safe_load(f)

# Make sure Kaggle uses the V2 dataset location
data["path"] = "/kaggle/input/datasets/jharshin/v2-enhanced-dataset/enhanced_yolo_dataset_v2"
data["train"] = "images/train"
data["val"] = "images/val"
data["test"] = "images/test"

# Save
with open(v2_yaml, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("V2 YAML created:")
print(v2_yaml)

print("\nDataset configuration:")
print(data)

In [ ]:
# verify dataset image and label counts
from pathlib import Path

v2_root = Path(
    "/kaggle/input/datasets/jharshin/v2-enhanced-dataset/enhanced_yolo_dataset_v2"
)

for split in ["train", "val", "test"]:
    images = v2_root / "images" / split
    labels = v2_root / "labels" / split

    print(
        split,
        "images =", len(list(images.iterdir())),
        "labels =", len(list(labels.iterdir()))
    )

In [ ]:
# sanity-check forward pass
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"

model_v4.to(device)

dummy = torch.randn(
    1, 3, 640, 640,
    device=device
)

with torch.no_grad():
    output = model_v4.model(dummy)

print("Forward pass successful.")
print("Device:", device)

### Load pretrained weights (index-aware transfer)

In [ ]:
# load pretrained weights (index-aware transfer)
import re

from ultralytics import YOLO

INSERTED_LAYERS = {5, 8, 11}

pretrained_sd = YOLO("yolov8s.pt").model.state_dict()
target_sd = model_v4.model.state_dict()


def stock_key(new_key):
    match = re.match(r"model\.(\d+)\.(.*)", new_key)
    if not match:
        return None
    new_idx, rest = int(match.group(1)), match.group(2)
    if new_idx in INSERTED_LAYERS:
        return None  # brand-new module, no pretrained counterpart
    offset = sum(1 for idx in INSERTED_LAYERS if idx < new_idx)
    return f"model.{new_idx - offset}.{rest}"


remapped = {}
for key, tensor in target_sd.items():
    old_key = stock_key(key)
    if old_key is not None and old_key in pretrained_sd and pretrained_sd[old_key].shape == tensor.shape:
        remapped[key] = pretrained_sd[old_key]

target_sd.update(remapped)
model_v4.model.load_state_dict(target_sd)

print(f"Transferred {len(remapped)}/{len(target_sd)} items from pretrained weights (index-aware remap)")

### Focal loss classification head

Replaces only the BCE classification term inside `v8DetectionLoss` with focal loss via a monkey-patch of `__init__`; box and DFL losses are untouched.

In [ ]:
# define focal loss classification head
import torch
import torch.nn as nn
import torch.nn.functional as F


class FocalBCEWithLogitsLoss(nn.Module):
    """
    BCEWithLogitsLoss with Focal Loss modulation.

    Only the YOLO classification loss is replaced.
    Box loss and DFL loss remain unchanged.
    """

    def __init__(self, gamma=2.0, alpha=0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, pred, target):
        # Standard BCE loss, unreduced
        bce = F.binary_cross_entropy_with_logits(
            pred,
            target,
            reduction="none"
        )

        # Convert logits to probabilities
        prob = torch.sigmoid(pred)

        # Probability assigned to the correct class
        p_t = target * prob + (1 - target) * (1 - prob)

        # Alpha weighting
        alpha_t = target * self.alpha + (1 - target) * (1 - self.alpha)

        # Focal modulation
        focal_weight = alpha_t * (1 - p_t).pow(self.gamma)

        return focal_weight * bce


print("Focal Loss class created successfully.")

In [ ]:
# patch yolov8 detection loss to use focal loss
from ultralytics.utils.loss import v8DetectionLoss


# Save the original initialization
_original_v8DetectionLoss_init = v8DetectionLoss.__init__


def focal_detection_loss_init(self, model, tal_topk=10, tal_topk2=None):
    """
    Initialize the normal YOLOv8 detection loss,
    then replace only its BCE classification loss with Focal Loss.
    """

    # Run the original YOLOv8 loss initialization
    _original_v8DetectionLoss_init(
        self,
        model,
        tal_topk=tal_topk,
        tal_topk2=tal_topk2
    )

    # Replace BCE with Focal BCE
    # gamma=2.0, alpha=0.25 are the defaults from the original RetinaNet paper
    self.bce = FocalBCEWithLogitsLoss(
        gamma=2.0,
        alpha=0.25
    )


# Monkey-patch the YOLOv8 detection loss
v8DetectionLoss.__init__ = focal_detection_loss_init

print("YOLOv8 detection loss patched successfully.")
print("Classification loss: Focal BCE")
print("gamma = 2.0")
print("alpha = 0.25")
print("Box loss: unchanged")
print("DFL loss: unchanged")

In [ ]:
# verify the loss patch was applied
from ultralytics.utils.loss import v8DetectionLoss

print("Checking patched loss...")

print("v8DetectionLoss.__init__:", v8DetectionLoss.__init__)

print("\nFocal Loss configuration:")
print("gamma =", 2.0)
print("alpha =", 0.25)

print("\nCA-CBAM model:")
print(model_v4.model)

print("\nVerification complete.")

### Train V4 model on V2 data

In [ ]:
# train the model
results_v4_focal = model_v4.train(
    data="/kaggle/working/datav2.yaml",

    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    patience=20,  # stop early if val mAP hasn't improved in 20 epochs
    seed=42,  # fixed seed for a reproducible, comparable run
    project="/kaggle/working/ntlnp_experiments",
    name="v4_yolov8s_ca_cbam_focal_v2data",
    pretrained=False,
    plots=True,
    save=True,
    verbose=True
)

### Evaluate best checkpoint

In [ ]:
# load the best checkpoint
best_model_v4 = YOLO("/kaggle/working/ntlnp_experiments/v4_yolov8s_ca_cbam_focal_v2data/weights/best.pt")

print("Best V4 model loaded.")

In [ ]:
# evaluate on validation set
val_results_v4 = best_model_v4.val(
    data="/kaggle/working/datav2.yaml",
    split="val"
)

print("V4 Validation mAP50:", val_results_v4.box.map50)
print("V4 Validation mAP50-95:", val_results_v4.box.map)
print("V4 Validation Precision:", val_results_v4.box.mp)
print("V4 Validation Recall:", val_results_v4.box.mr)

In [ ]:
# evaluate on test set
test_results_v4 = best_model_v4.val(
    data="/kaggle/working/datav2.yaml",
    split="test"
)

print("V4 Test mAP50:", test_results_v4.box.map50)
print("V4 Test mAP50-95:", test_results_v4.box.map)
print("V4 Test Precision:", test_results_v4.box.mp)
print("V4 Test Recall:", test_results_v4.box.mr)

In [ ]:
# predictions on test set
prediction_results_v4 = best_model_v4.predict(
    source="/kaggle/input/datasets/jharshin/v2-enhanced-dataset/enhanced_yolo_dataset_v2/images/test",
    conf=0.25,
    save=True
)

In [ ]:
# print per-class AP
names = best_model_v4.names

for i, name in names.items():
    print(
        f"{name:<15} "
        f"AP50: {test_results_v4.box.ap50[i]:.4f} "
        f"AP50-95: {test_results_v4.box.ap[i]:.4f}"
    )

### Compile & save metrics

In [ ]:
# compile evaluation results
metrics_v4 = {
    "Model": "v4_yolov8s_ca_cbam_focal_v2data",
    "Validation mAP50": val_results_v4.box.map50,
    "Validation mAP50-95": val_results_v4.box.map,
    "Validation Precision": val_results_v4.box.mp,
    "Validation Recall": val_results_v4.box.mr,
    "Test mAP50": test_results_v4.box.map50,
    "Test mAP50-95": test_results_v4.box.map,
    "Test Precision": test_results_v4.box.mp,
    "Test Recall": test_results_v4.box.mr,
}

print(metrics_v4)

In [ ]:
# save metrics as CSV
import pandas as pd
import os

results_file = "/kaggle/working/ntlnp_experiments/experiment_results_v4.csv"

df = pd.DataFrame([metrics_v4])

if os.path.exists(results_file):
    old = pd.read_csv(results_file)
    df = pd.concat([old, df], ignore_index=True)

df.to_csv(results_file, index=False)

print(df)

In [ ]:
# package results for download
!zip -r kaggle_working_v4.zip /kaggle/working